## Multi-energy system scheduling

### Simple formulation: ideal grid

In [ ]:
# pip install --upgrade nbstripout
# pip install pyomo gurobipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyomo.environ as pyo
import os
from pyomo.environ import *
from pyomo.opt import SolverFactory
from pyomo.core import Var

os.environ["GRB_LICENSE_FILE"] = os.environ["HOME"] + "/gurobi/gurobi.lic"

#### Data gathering

In [ ]:
#------------------------------------------------------------------------------------------------
# PARAMETERS
# (later they will be redefined to accomodate the typical Pyomo problem formulation)
#------------------------------------------------------------------------------------------------

# GENERAL ---------------------------------------------------------------------------------------
# FCR service
DT_FCR_ramp = 30 / 3600     # time allotted to reach allocated power in FCR service
C_FCR_capup = 10            # remuneration rate for upward FCR capacity bid [€/MW] ### NO REFERENCE ###
C_FCR_capdown = 10          # remuneration rate for downward FCR capacity bid [€/MW] ### NO REFERENCE ###

# H2 demand
C_h2 = 7      # H2 selling price [€/kg]
Hsup_min_day = 25          # minimum daily hydrogen supply [kg] ### TO DECIDE ###
Hsup_max_day = 50          # maximum daily hydrogen supply [kg] ### TO DECIDE ###
Hsup_tgt_week = (Hsup_max_day + Hsup_min_day) / 2 * 7      # target weekly H2 supply (evaluated as daily mean times 7 days) ### TO DECIDE ###
Hsup_tgt_month = (Hsup_max_day + Hsup_min_day) / 2 * 30    # target monthy H2 supply (evaluated as daily mean times 30 days) ### TO DECIDE ###

# LOADS -----------------------------------------------------------------------------------------
Pl_comm_rtd = 1     # rated commertial load power [MW]
Pl_ind_rtd = 1      # rated industrial load power [MW]
Pl_resid_rtd = 1    # rated residential load power [MW]

# PV --------------------------------------------------------------------------------------------
Ppv_rtd = 3    # PV rated power [MW]

# BESS ------------------------------------------------------------------------------------------
Pb_rtd = 1              # BESS rated power [MW]
Pb_ch_max = Pb_rtd      # BESS maximum charging power [MW]
Pb_dch_max = Pb_rtd     # BESS maximum discharging power [MW]
C_rate = 0.5            # BESS C-rate [1/h]
Eb_rtd = Pb_rtd / C_rate    # BESS rated capacity [MWh]
DoD = 0.9               # BESS depth of discharge
SoC_max = 1             # maximum SoC level
SoC_min = 1 - DoD       # minimum SoC level
SoC_0 = 0.5 * SoC_max   # SoC value at t=0
eta_b_rt = 0.931        # battery round-trip efficiency
eta_b_inv_rt = 0.98     # inverter round-trip efficiency
eta_bch = np.sqrt(eta_b_rt)*np.sqrt(eta_b_inv_rt)       # BESS charging efficiency
eta_bdch = np.sqrt(eta_b_rt)*np.sqrt(eta_b_inv_rt)      # BESS discharging efficiency
sig_b_sd = 0.01          # BESS hourly self-discharge coefficient [-] ### NO REFERENCE ###

# HSS -------------------------------------------------------------------------------------------
H_hss_max = 1e3                 # hydrogen tank capacity [kg]
H_hss_min = 0.1 * H_hss_max     # minimum hydrogen tank level [kg]
H_hss_0 = 0.5 * H_hss_max       # H2 level at t=0
FR_hc_el2hss = 25               # H2 maximum flow rate for the compressor from EL to HSS ### NO REFERENCE ###
FR_hc_hss2dem = 25              # H2 maximum flow rate for the compressor from EL to HRS (H2 demand) ### NO REFERENCE ###

sig_hss_sd = 0.0002         # HSS hourly H2 self-discharge coefficient [-] ### NO REFERENCE ###
sig_hss_ch = 0              # HSS hourly H2 charging losses coefficient [-]
sig_hss_dch = 0             # HSS hourly H2 discharging losses coefficient [-]
sig_hc_el2hss = 0           # coefficient of H2 losses for EL to HSS compression stage [-]
sig_hc_hss2dem = 0          # coefficient of H2 losses for HSS to HRS compression stage [-]

# FC --------------------------------------------------------------------------------------------
lhv = 33.3e-3       # lower heating value of H2 [MWh/kg]
eta_fc = 0.4        # FC efficiency ### NO REFERENCE ###

Pfc_rtd = 0.5          # FC rated power [MW]
Pfc_min_ratio = 0.1    # FC minimum power ratio [-] ### NO REFERENCE ###
Pfc_min = Pfc_min_ratio * Pfc_rtd       # FC minimum turndown level [MW] (also minimum power level)
rho_fcpos = 20 * Pfc_rtd    # FC positive ramp rate [MW/h] ### NO REFERENCE ###
rho_fcneg = -20 * Pfc_rtd   # FC negative ramp rate [MW/h] ### NO REFERENCE ###

# COMPRESSORS -----------------------------------------------------------------------------------
C_hc = 0.001        # Energy requirement for the compression of a hydrogen mass unit [MWh/kg] ### NO REFERENCE ###

In [ ]:
#------------------------------------------------------------------------------------------------
# EL MODEL AND PIECEWISE LINEAR APPROXIMATION
#------------------------------------------------------------------------------------------------


# ORIGINAL MODEL --------------------------------------------------------------------------------
# General constants and parameters
T0 = 20 + 273.15    # standard temperature [K]
p0 = 1.01325e5      # standard pressure [Pa]
R = 8.3145          # universal gas constant [J/(mol*K)]
F = 96485           # Faraday constant [C/mol]
z = 2               # number of electrons transferred in electrolysis reaction
Mm = 2.01568e-3     # molar mass of hydrogen [kg/mol]

# Electrolyzer parameters
p = 30e5            # operating pressure [Pa]
T = 70 + 273.15     # operating temperature [K]
R_io = 0.326        # internal stack resistance at standard conditions [Ohm]
k = 0.0395          # fitting parameter
dR_t = -3.812e-3    # temperature coefficient of resistance [Ohm/K]
e_rev0 = 1.476      # reversible voltage at standard conditions [V]
eta_F = 1           # Faraday efficiency

Pel_rtd_W = 1e6         # EL rated power [W]
Pel_min_ratio = 0.1     # EL minimum power ratio [-]
Pel_min_W = Pel_min_ratio * Pel_rtd_W     # EL minimum power [W]
n_s = 275           # number of cells in series
n_p = 3700          # number of cells in parallel
n_tot = n_s * n_p   # total number of cells

# (Empirical) Electrical model
V_rev = e_rev0 + (R*T)/(2*F) * np.log(p/p0)         # reversible voltage [V]
Ri = R_io + k * np.log(p/p0) + dR_t * (T - T0)      

Pel_fit = np.linspace(Pel_min_W, Pel_rtd_W, 901)  # power array for fitting [W]
Pel_fit_cell = Pel_fit / n_tot           # cell power array for fitting [W]

i_fit = (-V_rev + np.sqrt(V_rev**2 + 4*Ri*Pel_fit_cell)) / (2*Ri)      # current array [A]
H2_prod_fit = eta_F * Mm * 3600 * (n_tot * i_fit) / (z * F)            # H2 production rate array for fitting [kg/h]



# PIECEWISE LINEAR MODEL (for on state) (3 segments) -----------------------------------------------------------
H2_prod_reduce = H2_prod_fit[::10]
Pel_reduce = Pel_fit[::10]

# Search the optimal breakpoint for the piecewise linear approximation
n_break = 2    # number of breakpoints for the pwl model (x breakpoints => x+1 segments)

opt_slope_s1 = 0
opt_slope_s2 = 0
opt_slope_s3 = 0
opt_intercept_s1 = 0
opt_intercept_s2 = 0
opt_intercept_s3 = 0
abs_err_best = np.inf
pwl_breakpoint = 0

for p1 in range(1, len(Pel_reduce) - n_break):
    slope_s1 = (H2_prod_reduce[p1] - H2_prod_reduce[0]) / (Pel_reduce[p1] - Pel_reduce[0])    # slope of the first segment
    intercept_s1 = H2_prod_reduce[0] - slope_s1 * Pel_reduce[0]                               # intercept of the first segment
    for p2 in range(p1 + 1, len(Pel_reduce) - n_break + 1):
        slope_s2 = (H2_prod_reduce[p2] - H2_prod_reduce[p1]) / (Pel_reduce[p2] - Pel_reduce[p1])  # slope of the second segment
        intercept_s2 = H2_prod_reduce[p1] - slope_s2 * Pel_reduce[p1]
        slope_s3 = (H2_prod_reduce[-1] - H2_prod_reduce[p2]) / (Pel_reduce[-1] - Pel_reduce[p2])  # slope of the third segment
        intercept_s3 = H2_prod_reduce[p2] - slope_s3 * Pel_reduce[p2]
        
        abs_err = sum(abs(H2_prod_fit[0:p1*10] - (slope_s1 * Pel_fit[0:p1*10] + intercept_s1))) + \
          sum(abs(H2_prod_fit[p1*10:p2*10] - (slope_s2 * Pel_fit[p1*10:p2*10] + intercept_s2))) + \
          sum(abs(H2_prod_fit[p2*10:] - (slope_s3 * Pel_fit[p2*10:] + intercept_s3)))
        # print(abs_err)

        if abs_err < abs_err_best:
            abs_err_best = abs_err
            opt_slope_s1 = slope_s1
            opt_slope_s2 = slope_s2
            opt_slope_s3 = slope_s3
            opt_intercept_s1 = intercept_s1
            opt_intercept_s2 = intercept_s2
            opt_intercept_s3 = intercept_s3
            pwl_breakpoint_p1 = p1*10
            pwl_breakpoint_p2 = p2*10
            # print(f"Last best breackpoint indexes = {pwl_breakpoint_p1}, {pwl_breakpoint_p2}")

            
# Print results
print(f"Best breakpoint indexes: {pwl_breakpoint_p1}, {pwl_breakpoint_p2} \n Absolute error = {abs_err_best}")
print(f"They correspond to the power values: {Pel_fit[pwl_breakpoint_p1]} W and {Pel_fit[pwl_breakpoint_p2]} W")
print(f"Slopes and intecepts of the segments are are: \
      \n First segment slope = {opt_slope_s1}, intercept = {opt_intercept_s1}) \
      \n Second segment slope = {opt_slope_s2}, intercept = {opt_intercept_s2} \
      \n Third segment slope = {opt_slope_s3}, intercept = {opt_intercept_s3}")

# H2 production according to the pwl model (for plotting)
H2_prod_pwl_s1 = opt_slope_s1 * Pel_fit[0:pwl_breakpoint_p1] + opt_intercept_s1
H2_prod_pwl_s2 = opt_slope_s2 * Pel_fit[pwl_breakpoint_p1:pwl_breakpoint_p2] + opt_intercept_s2
H2_prod_pwl_s3 = opt_slope_s3 * Pel_fit[pwl_breakpoint_p2:] + opt_intercept_s3
H2_prod_pwl = np.concatenate((H2_prod_pwl_s1, H2_prod_pwl_s2, H2_prod_pwl_s3))


# Adjust parameters to express EL power in MW
Pel_rtd = Pel_rtd_W / 1e6                  # EL rated power [MW] (also maximum power)
Pel_min = Pel_min_ratio * Pel_rtd     # EL minimum turndown level [MW] (also minimum power level)
a_s1 = opt_slope_s1 * 1e6
a_s2 = opt_slope_s2 * 1e6
a_s3 = opt_slope_s3 * 1e6
b_s1 = opt_intercept_s1
b_s2 = opt_intercept_s2
b_s3 = opt_intercept_s3
bp_s12 = Pel_fit[pwl_breakpoint_p1] / 1e6   # breakpoint power level between segments 1 and 2 [MW]
bp_s23 = Pel_fit[pwl_breakpoint_p2] / 1e6   # breakpoint power level between segments 2 and 3 [MW]
s1_start, s1_end = Pel_min, bp_s12    # lower and upper power boundaries of segment 1
s2_start, s2_end = bp_s12, bp_s23
s3_start, s3_end = bp_s23, Pel_rtd


# ADDITIONAL PARAMETERS ----------------------------------------------------------------------------------------
Pel_sb = 0.02 * Pel_rtd    # EL stand-by power consumption [MW] ### NO REFERENCE ###
rho_elpos = 20 * Pel_rtd    # EL positive ramp rate [MW/h] ### NO REFERENCE ###
rho_elneg = -20 * Pel_rtd   # EL negative ramp rate [MW/h] ### NO REFERENCE ###
print(s1_start)
s1_end

In [ ]:
#------------------------------------------------------------------------------------------------
# PROFILES
#------------------------------------------------------------------------------------------------

# Reading the Excel files
C_ele_original = pd.read_excel('GME_20240101_20241231_MGP_PrezziZonali_CSUD.xlsx', decimal=',')
# print(C_ele_original)
Pl_Ppv_Pwt_original = pd.read_excel('Dati ENEA.xlsx')
# print(Pl_Ppv_Pwt_original)


# Extraction of useful profiles
C_ele_full = C_ele_original['€/MWh'].tolist()               # zonal price (CSUD 2024) [€/MWh], from 01/01/24 to 31/12/24 (leap year, 366 days)
C_ele = np.array(C_ele_full[:1416] + C_ele_full[1440:])     # removing 29th Feb data to have 365 days (8760 hours) and turning into ndarray
# C_ele = np.append(C_ele, C_ele[0])                        # make the array go from 0 to 8760 (8761 values)

Pl_comm_to_arrange = Pl_Ppv_Pwt_original['Commerciale'].tolist()            # commercial load profile [p.u.], from 16/10/20 to 15/10/21
Pl_comm = np.array(Pl_comm_to_arrange[1848:] + Pl_comm_to_arrange[:1848])   # rearranged data from 01/01/21 to 15/10/21 and then from 16/10/20 to 31/12/20
# Pl_comm = np.append(Pl_comm, Pl_comm[0])

Pl_ind_to_arrange = Pl_Ppv_Pwt_original['Industriale'].tolist()             # industrial load profile [p.u.], from 16/10/20 to 15/10/21
Pl_ind = np.array(Pl_ind_to_arrange[1848:] + Pl_ind_to_arrange[:1848])
# Pl_ind = np.append(Pl_ind, Pl_ind[0])

Pl_resid_to_arrange = Pl_Ppv_Pwt_original['Residenziale'].tolist()          # residential load profile [p.u.], from 16/10/20 to 15/10/21
Pl_resid = np.array(Pl_resid_to_arrange[1848:] + Pl_resid_to_arrange[:1848])
# Pl_resid = np.append(Pl_resid, Pl_resid[0])

Ppv_to_arrange = Pl_Ppv_Pwt_original['PV'].tolist()         # PV generation profile [??], from 16/10/20 to 15/10/21
P_pv = np.array(Ppv_to_arrange[1848:] + Ppv_to_arrange[:1848])
# P_pv = np.append(P_pv, P_pv[0])

Pwt_to_arrange = Pl_Ppv_Pwt_original['Wind'].tolist()       # WT generation profile [??], from 16/10/20 to 15/10/21
P_wt = np.array(Pwt_to_arrange[1848:] + Pwt_to_arrange[:1848])
# P_wt = np.append(P_wt, P_wt[0])


# Checking the data
time = np.arange(0, 8760)

# Plot settings
plt.rc('figure', figsize=(20, 20))
plt.rc('font', family='monospace', weight='bold', size=11)
plt.rc('axes', labelsize=11, titlesize=14)
plt.rc('legend', fontsize=11)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)
plt.rcParams['axes.grid'] = True
plt.rcParams['lines.linewidth'] = 2

# Plots
fig, axs = plt.subplots(6, 1, layout='constrained')
axs[0].plot(time, C_ele, color='blue')
axs[0].set_xlabel('x label')
axs[0].set_ylabel('y label')
axs[0].set_title("Electricity Price [€/MWh]")
axs[0].legend()

axs[1].plot(time, Pl_comm)
axs[1].set_title("Commercial load [p.u.]")
axs[2].plot(time, Pl_ind)
axs[2].set_title("Industrial load [p.u.]")
axs[3].plot(time, Pl_resid)
axs[3].set_title("Residential load [p.u.]")
axs[4].plot(time, P_pv)
axs[4].set_title("PV generation [??]")
axs[5].plot(time, P_wt)
axs[5].set_title("WT generation [??]")

In [ ]:
# Profiles adjustments based on rated powers
Pl_comm = Pl_comm * Pl_comm_rtd         # Commercial load profile [MW]
Pl_ind = Pl_ind * Pl_ind_rtd            # Industrial load profile [MW]
Pl_resid = Pl_resid * Pl_resid_rtd      # Residential load profile [MW]
P_pv = (P_pv / 2) * Ppv_rtd             # PV generation profile [MW]
# Pwt = Pwt * Pwt_rtd                   # WT generation profile [MW]

# Checking the data after adjustments
fig, axs = plt.subplots(6, 1, layout='constrained')
axs[0].plot(time, C_ele, color='blue')
axs[0].set_xlabel('x label')
axs[0].set_ylabel('y label')
axs[0].set_title("Electricity Price [€/MWh]")
axs[0].legend()

axs[1].plot(time, Pl_comm)
axs[1].set_title("Commercial load [MW]")
axs[2].plot(time, Pl_ind)
axs[2].set_title("Industrial load [MW]")
axs[3].plot(time, Pl_resid)
axs[3].set_title("Residential load [MW]")
axs[4].plot(time, P_pv)
axs[4].set_title("PV generation [MW]")
# axs[5].plot(time, P_wt)
# axs[5].set_title("WT generation")

In [ ]:
#------------------------------------------------------------------------------------------------
# PROBLEM DEFINITION IN PYOMO (1/4)
#------------------------------------------------------------------------------------------------


# GENERAL PROBLEM PARAMETERS
time_window = 24 * 7 * 52           # time window of simulation [h]
D_t = 1                         # timestep [h]
n_days = time_window / 24       # number of days simulated [d]
n_weeks = time_window / 24 / 7  # number of weeks simulated [w]


#------------------------------------------------------------------------------------------------
# MODEL
#------------------------------------------------------------------------------------------------

m = pyo.ConcreteModel()


#------------------------------------------------------------------------------------------------
# SETS
#------------------------------------------------------------------------------------------------

m.TIME = pyo.RangeSet(0, time_window - 1, 1)         # Set of the timesteps
m.RES = pyo.Set(initialize = ['PV'])                 # Set of renewable generation units
m.BESS = pyo.Set(initialize = ['BESS'])              # Set of BESSs
m.EL = pyo.Set(initialize = ['EL'])                  # Set of electrolyzers
m.ELpwl = pyo.Set(initialize = ['s1', 's2', 's3'])   # Set of the segments for the pwl EL model
m.HSS = pyo.Set(initialize = ['HSS'])                # Set of hydrogen storage systems
m.FC = pyo.Set(initialize = ['FC'])                  # Set of fuel cells
# m.LOAD = pyo.Set(initialize = ['L1', 'L2'])          # Set of non-controllable loads
m.DAYS = pyo.RangeSet(0, n_days - 1, 1)              # Set of days
m.WEEKS = pyo.RangeSet(0, n_weeks - 1, 1)            # Set of weeks


#------------------------------------------------------------------------------------------------
# PARAMETERS
#------------------------------------------------------------------------------------------------

# RES -------------------------------------------------------------------------------------------


# LOADS -----------------------------------------------------------------------------------------
# Create a dictionary with load - profiles correspondences
# (I'm not sure this is the best/cleanest approach. In the future I will also have to assign bus numbers to these objects.)
# Pd_dict = {}
# for t in range(time_window):
#     Pd_dict[('L1', t)] = Pl_ind[t]
#     Pd_dict[('L2', t)] = Pl_comm[t]

# Pd = pyo.Param(m.LOAD, m.TIME, initialize = Pd_dict)

# # Checking the values
# print(Pd_dict)
# print(Pd_dict['L1', 1])
# # If I want to access all the elements associated with L1:
# L1_values = [v for (load, t), v in Pd_dict.items() if load == 'L1']
# # Including the key (which in this case is the timestep index):
# L1_series = [(t, v) for (load, t), v in Pd_dict.items() if load == 'L1']
# print(L1_values)
# print(L1_series)


# BESS ------------------------------------------------------------------------------------------
Pb_ch_max = {'BESS': Pb_rtd}
Pb_dch_max = {'BESS': Pb_rtd}
Eb_rtd = {'BESS': Pb_rtd / C_rate}
SoC_max = {'BESS': 1}
SoC_min = {'BESS': 1 - DoD}
SoC_0 = {'BESS': 0.5 * 1}
eta_bch = {'BESS': np.sqrt(eta_b_rt)*np.sqrt(eta_b_inv_rt)}
eta_bdch = {'BESS': np.sqrt(eta_b_rt)*np.sqrt(eta_b_inv_rt)}
sig_b_sd = {'BESS': 0.001}    ### NO REFERENCE ###


# EL --------------------------------------------------------------------------------------------
Pel_rtd = {'EL': Pel_rtd_W / 1e6}
Pel_min = {'EL': Pel_min_ratio * Pel_rtd_W / 1e6}
Pel_sb = {'EL': 0.02 * Pel_rtd_W / 1e6}     ### NO REFERENCE ###
rho_elpos = {'EL': 20 * Pel_rtd_W / 1e6}     ### NO REFERENCE ###
rho_elneg = {'EL': -20 * Pel_rtd_W / 1e6}     ### NO REFERENCE ###

A = {'s1': a_s1, 's2': a_s2, 's3': a_s3}    # slopes of the segments of the pwl EL model
B = {'s1': b_s1, 's2': b_s2, 's3': b_s3}    # intercepts of the segments of the pwl EL model
Pseg_low = {'s1': s1_start, 's2': s2_start, 's3': s3_start}     # lower power boundaries of the pwl segments
Pseg_high = {'s1': s1_end, 's2': s2_end, 's3': s3_end}          # upper power boundaries of the pwl segments


# HSS -------------------------------------------------------------------------------------------
H_hss_max = {'HSS': 5e3}
H_hss_min = {'HSS': 0.1 * 5e3}
H_hss_0 = {'HSS': 0.5 * 5e3}
FR_hc_el2hss = {'HSS': 25}
FR_hc_hss2dem = {'HSS': 25}

sig_hss_sd = {'HSS': 0.00002}     ### NO REFERENCE ###
sig_hss_ch = {'HSS': 0}
sig_hss_dch = {'HSS': 0}
sig_hc_el2hss = {'HSS': 0}
sig_hc_hss2dem = {'HSS': 0}


# FC --------------------------------------------------------------------------------------------
eta_fc = {'FC': 0.5}    ### NO REFERENCE ###
Pfc_rtd = {'FC': 0.5}
Pfc_min = {'FC': Pfc_min_ratio * 0.5}
rho_fcpos = {'FC': 20 * 0.5}    ### NO REFERENCE ###
rho_fcneg = {'FC': -20 * 0.5}    ### NO REFERENCE ###


# PARAMETERS WRAPPING IN PYOMO "Param" objects --------------------------------------------------
m.Dt = pyo.Param(initialize = D_t)

m.C_ele = pyo.Param(m.TIME, initialize = C_ele[:time_window], domain = pyo.NonNegativeReals)
m.Ppv = pyo.Param(m.TIME, initialize = P_pv[:time_window], domain = pyo.NonNegativeReals)
m.Pd_ind = pyo.Param(m.TIME, initialize = Pl_ind[:time_window], domain = pyo.NonNegativeReals)
m.Pd_comm = pyo.Param(m.TIME, initialize = Pl_comm[:time_window], domain = pyo.NonNegativeReals)

m.DT_FCR_ramp = pyo.Param(initialize = 30 / 3600, domain = pyo.NonNegativeReals)
m.C_FCR_capup = pyo.Param(initialize = 10, domain = pyo.NonNegativeReals)    ### NO REFERENCE ###
m.C_FCR_capdown = pyo.Param(initialize = 10, domain = pyo.NonNegativeReals)    ### NO REFERENCE ###

m.C_h2 = pyo.Param(initialize = 7, domain = pyo.NonNegativeReals)    ### TO DECIDE ###
m.Hsup_min_day = pyo.Param(initialize = 25, domain = pyo.NonNegativeReals)    ### TO DECIDE ###
m.Hsup_max_day = pyo.Param(initialize = 50, domain = pyo.NonNegativeReals)    ### TO DECIDE ###
m.Hsup_tgt_week = pyo.Param(initialize = (25 + 50) / 2 * 7, domain = pyo.NonNegativeReals)      ### TO DECIDE ###
m.Hsup_tgt_month = pyo.Param(initialize = (25 + 50) / 2 * 30, domain = pyo.NonNegativeReals)    ### TO DECIDE ###

m.lhv = pyo.Param(initialize = 33.3e-3, domain = pyo.NonNegativeReals)

m.C_hc = pyo.Param(initialize = 0.001, domain = pyo.NonNegativeReals)    ### NO REFERENCE ###

m.Pb_ch_max = pyo.Param(m.BESS, initialize = Pb_ch_max, domain = pyo.NonNegativeReals)
m.Pb_dch_max = pyo.Param(m.BESS, initialize = Pb_dch_max, domain = pyo.NonNegativeReals)
### Question: the data defined using natural numbers is automatically saved as "int". Is it ok or should I change the definitions to have floats?
###           For example, Pb_rtd could be changed from "1" to "1.0".
m.Eb_rtd = pyo.Param(m.BESS, initialize = Eb_rtd, domain = pyo.NonNegativeReals)
m.SoC_max = pyo.Param(m.BESS, initialize = SoC_max, domain = pyo.PercentFraction)
m.SoC_min = pyo.Param(m.BESS, initialize = SoC_min, domain = pyo.PercentFraction)
m.SoC_0 = pyo.Param(m.BESS, initialize = SoC_0, domain = pyo.PercentFraction)
m.eta_bch = pyo.Param(m.BESS, initialize = eta_bch, domain = pyo.PercentFraction)
m.eta_bdch = pyo.Param(m.BESS, initialize = eta_bdch, domain = pyo.PercentFraction)
m.sig_b_sd = pyo.Param(m.BESS, initialize = sig_b_sd, domain = pyo.PercentFraction)

m.Pel_rtd = pyo.Param(m.EL, initialize = Pel_rtd, domain = pyo.NonNegativeReals)
m.Pel_min = pyo.Param(m.EL, initialize = Pel_min, domain = pyo.NonNegativeReals)
m.Pel_sb = pyo.Param(m.EL, initialize = Pel_sb, domain = pyo.NonNegativeReals)
m.rho_elpos = pyo.Param(m.EL, initialize = rho_elpos, domain = pyo.NonNegativeReals)
m.rho_elneg = pyo.Param(m.EL, initialize = rho_elneg, domain = pyo.NonPositiveReals)

m.A = pyo.Param(m.ELpwl, initialize = A, domain = pyo.Reals)
m.B = pyo.Param(m.ELpwl, initialize = B, domain = pyo.Reals)
m.Pseg_low = pyo.Param(m.ELpwl, initialize = Pseg_low, domain = pyo.NonNegativeReals)
m.Pseg_high = pyo.Param(m.ELpwl, initialize = Pseg_high, domain = pyo.NonNegativeReals)

m.H_hss_max = pyo.Param(m.HSS, initialize = H_hss_max, domain = pyo.NonNegativeReals)
m.H_hss_min = pyo.Param(m.HSS, initialize = H_hss_min, domain = pyo.NonNegativeReals)
m.H_hss_0 = pyo.Param(m.HSS, initialize = H_hss_0, domain = pyo.NonNegativeReals)
m.FR_hc_el2hss = pyo.Param(m.HSS, initialize = FR_hc_el2hss, domain = pyo.NonNegativeReals)
m.FR_hc_hss2dem = pyo.Param(m.HSS, initialize = FR_hc_hss2dem, domain = pyo.NonNegativeReals)
m.sig_hss_sd = pyo.Param(m.HSS, initialize = sig_hss_sd, domain = pyo.PercentFraction)
m.sig_hss_ch = pyo.Param(m.HSS, initialize = sig_hss_ch, domain = pyo.PercentFraction)
m.sig_hss_dch = pyo.Param(m.HSS, initialize = sig_hss_dch, domain = pyo.PercentFraction)
m.sig_hc_el2hss = pyo.Param(m.HSS, initialize = sig_hc_el2hss, domain = pyo.PercentFraction)
m.sig_hc_hss2dem = pyo.Param(m.HSS, initialize = sig_hc_hss2dem, domain = pyo.PercentFraction)

m.eta_fc = pyo.Param(m.FC, initialize = eta_fc, domain = pyo.PercentFraction)
m.Pfc_rtd = pyo.Param(m.FC, initialize = Pfc_rtd, domain = pyo.NonNegativeReals)
m.Pfc_min = pyo.Param(m.FC, initialize = Pfc_min, domain = pyo.NonNegativeReals)
m.rho_fcpos = pyo.Param(m.FC, initialize = rho_fcpos, domain = pyo.NonNegativeReals)
m.rho_fcneg = pyo.Param(m.FC, initialize = rho_fcneg, domain = pyo.NonPositiveReals)

In [ ]:
#------------------------------------------------------------------------------------------------
# PROBLEM DEFINITION IN PYOMO (2/4)
#------------------------------------------------------------------------------------------------

#------------------------------------------------------------------------------------------------
# VARIABLES
#------------------------------------------------------------------------------------------------

m.pb_dch = pyo.Var(m.BESS, m.TIME, domain = pyo.NonNegativeReals, bounds = lambda m,b,t: (0, m.Pb_dch_max[b]))
### Question: is it ok to define bounds in this way? Or is it preferrable to do it with a "def" function?
m.pb_ch = pyo.Var(m.BESS, m.TIME, domain = pyo.NonNegativeReals, bounds = lambda m,b,t: (0, m.Pb_ch_max[b]))
m.soc = pyo.Var(m.BESS, m.TIME, domain = pyo.PercentFraction, bounds = lambda m,b,t: (m.SoC_min[b], m.SoC_max[b]))
m.xb = pyo.Var(m.BESS, m.TIME, domain = pyo.Binary)

m.pel = pyo.Var(m.EL, m.TIME, domain = pyo.NonNegativeReals, bounds = lambda m,e,t: (0, m.Pel_rtd[e]))
# Question: Is it better to try and add an upper bound? I could define one if it helps the solution.
m.xel_on = pyo.Var(m.EL, m.TIME, domain = pyo.Binary)
m.xel_off = pyo.Var(m.EL, m.TIME, domain = pyo.Binary)
m.xel_sb = pyo.Var(m.EL, m.TIME, domain = pyo.Binary)
m.xel_su = pyo.Var(m.EL, m.TIME, domain = pyo.Binary)
# In theory, the following variables should also depend on the set "EL", as to be generalized to the case in which we have multiple
# electrolyzers. However, as I know I will be considering only one EL, I omitted the dependency on m.EL.
# m.pe_st = pyo.Var(m.ELpwl, m.TIME, domain = pyo.NonNegativeReals, bounds = lambda m,s,t: (0, m.Pseg_high[s]))
m.pe_st = pyo.Var(m.ELpwl, m.TIME, domain = pyo.NonNegativeReals)
m.xe_st = pyo.Var(m.ELpwl, m.TIME, domain = pyo.Binary)
def el_h2_prod(m, e, t):
    return sum(m.A[s] * m.pe_st[s, t] + m.B[s] * m.xe_st[s, t] for s in m.ELpwl) * m.Dt
m.hel = pyo.Expression(m.EL, m.TIME, rule = el_h2_prod)

m.hhss = pyo.Var(m.HSS, m.TIME, domain = pyo.NonNegativeReals, bounds = lambda m,hs,t: (m.H_hss_min[hs], m.H_hss_max[hs]))
# In theory, I could consider multiple hydrogen consumers, in which case a set "m.DEM" would be defined and the following variable would
# depend on it as well.
m.hdem = pyo.Var(m.TIME, domain = pyo.NonNegativeReals, bounds = (0, 1e5)) ### trovare un valore più preciso (flow rate compressore)

m.pfc = pyo.Var(m.FC, m.TIME, domain = pyo.NonNegativeReals, bounds = lambda m,f,t: (0, m.Pfc_rtd[f]))
m.hfc = pyo.Expression(m.FC, m.TIME, rule = lambda m,f,t: m.pfc[f, t] / (m.lhv * m.eta_fc[f]))
m.xfc_on = pyo.Var(m.FC, m.TIME, domain = pyo.Binary)

m.pel_fcr_up = pyo.Var(m.EL, m.TIME, domain = pyo.NonNegativeReals, bounds = lambda m,e,t: (0, m.Pel_rtd[e]))
m.pel_fcr_dwn = pyo.Var(m.EL, m.TIME, domain = pyo.NonNegativeReals, bounds = lambda m,e,t: (0, m.Pel_rtd[e]))

m.pfc_fcr_up = pyo.Var(m.FC, m.TIME, domain = pyo.NonNegativeReals, bounds = lambda m,f,t: (0, m.Pfc_rtd[f]))
m.pfc_fcr_dwn = pyo.Var(m.FC, m.TIME, domain = pyo.NonNegativeReals, bounds = lambda m,f,t: (0, m.Pfc_rtd[f]))

m.pgrid = pyo.Var(m.TIME, domain = pyo.Reals)   # power from the main grid

In [ ]:
#------------------------------------------------------------------------------------------------
# PROBLEM DEFINITION IN PYOMO (3/4)
#------------------------------------------------------------------------------------------------

#------------------------------------------------------------------------------------------------
# CONSTRAINTS
#------------------------------------------------------------------------------------------------

# BESS MODEL CONSTRAINTS ------------------------------------------------------------------------
# BESS discharging power (pb_dch[t] <= Pb_dch_max)
def bess_dch_limits(m, b, t):
    return m.pb_dch[b, t] <= m.Pb_dch_max[b] * m.xb[b, t]
m.PbessDischarge = pyo.Constraint(m.BESS, m.TIME, rule = bess_dch_limits)

# BESS charging power (pb_ch[t] <= Pb_ch_max)
def bess_ch_limits(m, b, t):
    return m.pb_ch[b, t] <= m.Pb_ch_max[b] * (1 - m.xb[b, t])
m.PbessCharge = pyo.Constraint(m.BESS, m.TIME, rule = bess_ch_limits)

# BESS SoC modelling (SoC[0] = ...; SoC[t] = (1 - sig_b_sd) * SoC[t-1] + (1/Eb_rtd) * (eta_bch * pb_ch - pb_dch/eta_bdch) * Dt)
def soc_eval(m, b, t):
    if t == m.TIME.first():
        return m.soc[b, 0] == (1 - m.sig_b_sd[b]) * m.SoC_0[b] + (1 / m.Eb_rtd[b]) * (m.eta_bch[b] * m.pb_ch[b, 0] - m.pb_dch[b, 0] / m.eta_bdch[b]) * m.Dt
    else:
        return m.soc[b, t] == (1 - m.sig_b_sd[b]) * m.soc[b, t-1] + (1 / m.Eb_rtd[b]) * (m.eta_bch[b] * m.pb_ch[b, t] - m.pb_dch[b, t] / m.eta_bdch[b]) * m.Dt
m.SoCEvaluation = pyo.Constraint(m.BESS, m.TIME, rule = soc_eval)

# End SoC condition on the BESS (SoC[end] == SoC_0)
def soc_end(m, b):
    return m.soc[b, m.TIME.last()] == m.SoC_0[b]
m.SoCEnd = pyo.Constraint(m.BESS, rule = soc_end)



# EL MODEL CONSTRAINTS --------------------------------------------------------------------------
# EL working states selection (xel_on[t] + xel_off[t] + xel_sb[t] = 1)
def el_states(m, e, t):
    return m.xel_on[e, t] + m.xel_off[e, t] + m.xel_sb[e, t] == 1
m.ELStates = pyo.Constraint(m.EL, m.TIME, rule = el_states)

# Identification of EL startup (xel_su[0] = 0; xel_su[t] >= xel_off[t-1] + xel_on[t] + xel_sb[t] - 1)
def el_find_startup(m, e, t):
    if t == m.TIME.first():
        return m.xel_su[e, 0] == 0 
    else:
        return m.xel_su[e, t] >= m.xel_off[e, t-1] + m.xel_on[e, t] + m.xel_sb[e, t] - 1
m.ELFindStartup = pyo.Constraint(m.EL, m.TIME, rule = el_find_startup)

# Hydrogen-power EL piecewise linear characteristic   # REPLACED BY EXPRESSION
# def el_h2_prod(m, e, t):
#     return m.hel[e, t] == sum(m.A[s] * m.pe_st[s, t] + m.B[s] * m.xe_st[s, t] for s in m.ELpwl) * m.Dt
# m.ELH2Production = pyo.Constraint(m.EL, m.TIME, rule = el_h2_prod)

# EL pwl segments boundaries
def el_seg_bound_low(m, s, t):
    return m.Pseg_low[s] * m.xe_st[s, t] <= m.pe_st[s, t]
m.ELSegmentBoundLow = pyo.Constraint(m.ELpwl, m.TIME, rule = el_seg_bound_low)
def el_seg_bound_high(m, s, t):
    return m.pe_st[s, t] <= m.Pseg_high[s] * m.xe_st[s, t]
m.ELSegmentBoundHigh = pyo.Constraint(m.ELpwl, m.TIME, rule = el_seg_bound_high)
# Alternative formulation
# def el_seg_bounds(m, s, t):
#     return (m.Pseg_low[s] * m.xe_st[s, t], m.pe_st[s, t], m.Pseg_high[s] * m.xe_st[s, t]) 
# m.ELSegmentBounds = pyo.Constraint(m.ELpwl, m.TIME, rule = el_seg_bounds)

# Activation of a single segment at once of the EL pwl model (and only when the EL is on)
def el_seg_act(m, e, t):
    return sum(m.xe_st[s, t] for s in m.ELpwl) <= m.xel_on[e, t]
m.ELSegmentActivate = pyo.Constraint(m.EL, m.TIME, rule = el_seg_act)

# EL power evaluation
def el_power_eval(m, e, t):
    return m.pel[e, t] == m.Pel_sb[e] * m.xel_sb[e, t] + sum(m.pe_st[s, t] for s in m.ELpwl)
m.ELPowerEval = pyo.Constraint(m.EL, m.TIME, rule = el_power_eval)

# EL ramp rate constraints
def el_rampup(m, e, t):
    if t == m.TIME.last():
        return pyo.Constraint.Skip 
    return m.rho_elneg[e] <= m.pel[e, t+1] - m.pel[e, t]
m.ELrampup = pyo.Constraint(m.EL, m.TIME, rule = el_rampup)
def el_rampdown(m, e, t):
    if t == m.TIME.last():
        return pyo.Constraint.Skip
    return m.pel[e, t+1] - m.pel[e, t] <= m.rho_elpos[e]
m.ELrampdown = pyo.Constraint(m.EL, m.TIME, rule = el_rampdown)


# HSS MODEL CONSTRAINTS -------------------------------------------------------------------------
# Hydrogen tank level modelling
def h2_tank_level(m, hs, t):
    if t == m.TIME.first():
        return m.hhss[hs, 0] == (1 - m.sig_hss_sd[hs]) * m.H_hss_0[hs] + \
            (1 - m.sig_hss_ch[hs]) * (1 - m.sig_hc_el2hss[hs]) * sum(m.hel[e, 0] for e in m.EL) \
            - sum(m.hfc[f, 0] for f in m.FC) / (1 - m.sig_hss_dch[hs]) - m.hdem[0] / ((1 - m.sig_hss_dch[hs]) * (1 - m.sig_hc_hss2dem[hs]))
    else:
        return m.hhss[hs, t] == (1 - m.sig_hss_sd[hs]) * m.hhss[hs, t-1] + \
            (1 - m.sig_hss_ch[hs]) * (1 - m.sig_hc_el2hss[hs]) * sum(m.hel[e, t] for e in m.EL) \
            - sum(m.hfc[f, t] for f in m.FC) / (1 - m.sig_hss_dch[hs]) - m.hdem[t] / ((1 - m.sig_hss_dch[hs]) * (1 - m.sig_hc_hss2dem[hs]))
m.H2TankLevel = pyo.Constraint(m.HSS, m.TIME, rule = h2_tank_level)

# End hydrogen tank level condition
def h2_tank_level_end(m, hs):
    return m.hhss[hs, m.TIME.last()] == m.H_hss_0[hs]
m.H2TankLevelEnd = pyo.Constraint(m.HSS, rule = h2_tank_level_end)

# Maximum compressor flow rate el2hss
def max_comprflow_el2hss(m, hs, t):
    return sum(m.hel[e, t] for e in m.EL) <= m.FR_hc_el2hss[hs]
m.MaxComprFlowEl2hss = pyo.Constraint(m.HSS, m.TIME, rule = max_comprflow_el2hss)

# Maximum compressor flow rate hss2dem
def max_comprflow_hss2dem(m, hs, t):
    return m.hdem[t] / (1 - m.sig_hc_hss2dem[hs]) <= m.FR_hc_hss2dem[hs]
m.MaxComprFlowHss2dem = pyo.Constraint(m.HSS, m.TIME, rule = max_comprflow_hss2dem)


# FC MODEL CONSTRAINTS --------------------------------------------------------------------------
# Power-hydrogen linear model   # REPLACED BY EXPRESSION
# def fc_power(m, f, t):
#     return m.pfc[f, t] == m.eta_fc[f] * m.lhv * m.hfc[f, t]
# m.FCPower = pyo.Constraint(m.FC, m.TIME, rule = fc_power)

# FC on state power boundaries
def fc_on_bound_low(m, f, t):
    return m.Pfc_min[f] * m.xfc_on[f, t] <= m.pfc[f, t]
m.FCOnBoundLow = pyo.Constraint(m.FC, m.TIME, rule = fc_on_bound_low)
def fc_on_bound_high(m, f, t):
    return m.pfc[f, t] <= m.Pfc_rtd[f] * m.xfc_on[f, t]
m.FCOnBoundHigh = pyo.Constraint(m.FC, m.TIME, rule = fc_on_bound_high)

# FC ramp rate constraints
def fc_rampup(m, f, t):
    if t == m.TIME.last():
        return pyo.Constraint.Skip 
    return m.rho_fcneg[f] <= m.pfc[f, t+1] - m.pfc[f, t]
m.FCrampup = pyo.Constraint(m.FC, m.TIME, rule = fc_rampup)
def fc_rampdown(m, f, t):
    if t == m.TIME.last():
        return pyo.Constraint.Skip
    return m.pfc[f, t+1] - m.pfc[f, t] <= m.rho_fcpos[f]
m.FCrampdown = pyo.Constraint(m.FC, m.TIME, rule = fc_rampdown)


# FCR MODELLING ---------------------------------------------------------------------------------
# EL FCR participation limits based on power capacity
def el_fcr_caplim_down(m, e, t):
    return m.pel_fcr_dwn[e, t] <= m.Pel_rtd[e] * m.xel_on[e, t] - m.pel[e, t] 
m.ELFCRCapacityLimDown = pyo.Constraint(m.EL, m.TIME, rule = el_fcr_caplim_down)

def el_fcr_caplim_up(m, e, t):
    return m.pel_fcr_up[e, t] <= m.pel[e, t] - m.Pel_min[e] * m.xel_on[e, t]
m.ELFCRCapacityLimUp = pyo.Constraint(m.EL, m.TIME, rule = el_fcr_caplim_up)

# EL FCR participation limits based on ramp rate
def el_fcr_rampratelim_down(m, e, t):
    return m.pel_fcr_dwn[e, t] <= m.rho_elpos[e] * m.DT_FCR_ramp
m.ELFCRRamprateLimDown = pyo.Constraint(m.EL, m.TIME, rule = el_fcr_rampratelim_down)

def el_fcr_rampratelim_up(m, e, t):
    return m.pel_fcr_up[e, t] <= -m.rho_elneg[e] * m.DT_FCR_ramp
m.ELFCRRamprateLimUp = pyo.Constraint(m.EL, m.TIME, rule = el_fcr_rampratelim_up)

# EL FCR participation limit based on available renewable energy
def el_fcr_availab_pow(m, e, t):
    return m.pel_fcr_dwn[e, t] <= m.Ppv[t] + sum(m.pb_dch[b, t] - m.pb_ch[b, t] for b in m.BESS) - sum(m.pe_st[s, t] for s in m.ELpwl) \
                                   - (m.C_hc / m.Dt) * (m.hel[e, t] + m.hdem[t])
m.ELFCRAvailabPower = pyo.Constraint(m.EL, m.TIME, rule = el_fcr_availab_pow)

# EL FCR participation limit based on available hydrogen storage capacity
### DA RIDEFINIRE

# FC FCR participation limits based on power capacity
def fc_fcr_caplim_down(m, f, t):
    return m.pfc_fcr_dwn[f, t] <= m.pfc[f, t] - m.Pfc_min[f] * m.xfc_on[f, t]
m.FCFCRCapacityLimDown = pyo.Constraint(m.FC, m.TIME, rule = fc_fcr_caplim_down)

def fc_fcr_caplim_up(m, f, t):
    return m.pfc_fcr_up[f, t] <= m.Pfc_rtd[f] * m.xfc_on[f, t] - m.pfc[f, t] 
m.FCFCRCapacityLimUp = pyo.Constraint(m.FC, m.TIME, rule = fc_fcr_caplim_up)

# FC FCR participation limits based on ramp rate
def fc_fcr_rampratelim_down(m, f, t):
    return m.pfc_fcr_dwn[f, t] <= -m.rho_fcneg[f] * m.DT_FCR_ramp
m.FCFCRRamprateLimDown = pyo.Constraint(m.FC, m.TIME, rule = fc_fcr_rampratelim_down)

def fc_fcr_rampratelim_up(m, f, t):
    return m.pfc_fcr_up[f, t] <= m.rho_fcpos[f] * m.DT_FCR_ramp
m.FCFCRRamprateLimUp = pyo.Constraint(m.FC, m.TIME, rule = fc_fcr_rampratelim_up)

# FC FCR participation limit based on hydrogen availability
def fc_fcr_availab_h2(m, f, t):
    return m.pfc_fcr_up[f, t] / ((1-m.sig_hss_dch['HSS']) * m.eta_fc[f] * m.lhv) <= sum(m.hhss[hs, t] - m.H_hss_min[hs] for hs in m.HSS)
m.FCFCRAvailabH2 = pyo.Constraint(m.FC, m.TIME, rule = fc_fcr_availab_h2)


# SYSTEM CONSTRAINTS ----------------------------------------------------------------------------
# General power balance
def power_balance(m, t):
    return m.pgrid[t] + m.Ppv[t] - (m.Pd_ind[t] + m.Pd_comm[t]) + sum(m.pb_dch[b, t] - m.pb_ch[b,t] for b in m.BESS) \
    - sum(m.pel[e, t] for e in m.EL) + sum(m.pfc[f, t] for f in m.FC) \
    - (m.C_hc / m.Dt) * sum(m.hel[e, t] for e in m.EL) - (m.C_hc / m.Dt) * m.hdem[t] == 0
m.PowerBalance = pyo.Constraint(m.TIME, rule = power_balance)


# Green hydrogen constraint
def green_h2(m, t):
    return sum(m.pel[e, t] for e in m.EL) + sum(m.pb_ch[b, t] for b in m.BESS) \
    + (m.C_hc / m.Dt) * sum(m.hel[e, t] for e in m.EL) + (m.C_hc / m.Dt) * m.hdem[t] <= m.Ppv[t] + sum(m.pb_dch[b, t] for b in m.BESS)
m.GreenHydrogen = pyo.Constraint(m.TIME, rule = green_h2)


# HYDROGEN DEMAND CONSTRAINTS -------------------------------------------------------------------
# Daily limits on hydrogen supply
def h2_daily_supply_bound_low(m, d):
    return m.Hsup_min_day <= sum(m.hdem[d*24 + h] for h in range(24))
m.H2DailySupplyBoundLow = pyo.Constraint(m.DAYS, rule = h2_daily_supply_bound_low)
def h2_daily_supply_bound_high(m, d):
    return sum(m.hdem[d*24 + h] for h in range(24)) <= m.Hsup_max_day
m.H2DailySupplyBoundHigh = pyo.Constraint(m.DAYS, rule = h2_daily_supply_bound_high)

# Weekly / Monthly hydrogen supply target
def h2_target_supply(m, w):
    return sum(m.hdem[w*24*7 + h] for h in range(24*7)) == m.Hsup_tgt_week
m.H2TargetSupply = pyo.Constraint(m.WEEKS, rule = h2_target_supply)

In [ ]:
#------------------------------------------------------------------------------------------------
# PROBLEM DEFINITION IN PYOMO (4/4)
#------------------------------------------------------------------------------------------------

#------------------------------------------------------------------------------------------------
# OBJECTIVE FUNCTION
#------------------------------------------------------------------------------------------------

def tot_revenue(m):
    return sum(m.hdem[t] * m.C_h2 \
                + (m.Ppv[t] + sum(m.pb_dch[b, t] - m.pb_ch[b, t] for b in m.BESS) \
                - sum(m.pel[e, t] for e in m.EL) + sum(m.pfc[f, t] for f in m.FC)) * m.Dt * m.C_ele[t] \
                + sum(m.pel_fcr_up[e, t] * m.C_FCR_capup + m.pel_fcr_dwn[e, t] * m.C_FCR_capdown for e in m.EL) \
                + sum(m.pfc_fcr_up[f, t] * m.C_FCR_capup + m.pfc_fcr_dwn[f, t] * m.C_FCR_capdown for f in m.FC) for t in m.TIME)
m.Obj = pyo.Objective(rule = tot_revenue, sense = pyo.maximize)               


#------------------------------------------------------------------------------------------------
# OBJECTIVE FUNCTION
#------------------------------------------------------------------------------------------------

solver = pyo.SolverFactory("gurobi")  # change to "highs" if you don't have Gurobi
# res = solver.solve(m, tee=True)

from pyomo.environ import *
# Tell Gurobi to compute IIS if the model is infeasible
solver.options['IISMethod'] = 1  # optional, 1 is default
# Ask Gurobi to write the IIS
# Note: Pyomo does not have a direct computeIIS, you need to pass an option
solver.options['ResultFile'] = 'infeasible.ilp'
res = solver.solve(m, tee=True)


In [ ]:
#------------------------------------------------------------------------------------------------
# EXRACTING RESULTS
#------------------------------------------------------------------------------------------------

# res.write()

# # Method 1
# pel_vals = []  
# for t in m.TIME:
#     pel_vals.append(m.pel['EL', t].value)

# Method 2
C_ele_vals = np.array([m.C_ele[t] for t in m.TIME])
Ppv_vals = np.array([m.Ppv[t] for t in m.TIME])

pel_vals = np.array([m.pel[e, t].value for e in m.EL for t in m.TIME])
hel_vals = np.array([m.hel[e, t]() for e in m.EL for t in m.TIME])
xel_on_vals = [m.xel_on[e, t].value for e in m.EL for t in m.TIME]
xel_off_vals = [m.xel_off[e, t].value for e in m.EL for t in m.TIME]
xel_sb_vals = [m.xel_sb[e, t].value for e in m.EL for t in m.TIME]
xel_su_vals = [m.xel_su[e, t].value for e in m.EL for t in m.TIME]
pel_fcr_up_vals = np.array([m.pel_fcr_up[e, t].value for e in m.EL for t in m.TIME])
pel_fcr_dwn_vals = np.array([m.pel_fcr_dwn[e, t].value for e in m.EL for t in m.TIME])

pb_dch_vals = np.array([m.pb_dch[b, t].value for b in m.BESS for t in m.TIME])
pb_ch_vals = np.array([m.pb_ch[b, t].value for b in m.BESS for t in m.TIME])
pb_vals = pb_dch_vals - pb_ch_vals
# pb_vals = np.append(pb_vals[0], pb_vals)
soc_vals = [m.soc[b, t].value for b in m.BESS for t in m.TIME]
soc_vals = np.append(SoC_0['BESS'], soc_vals)   # adding the initial SoC value
xb_vals = np.array([m.xb[b, t].value for b in m.BESS for t in m.TIME])

hhss_vals = [m.hhss[hs, t].value for hs in m.HSS for t in m.TIME]
hhss_vals = np.append(H_hss_0['HSS'], hhss_vals)   # adding the initial hydrogen level value

hdem_vals = np.array([m.hdem[t].value for t in m.TIME])

pfc_vals = np.array([m.pfc[f, t].value for f in m.FC for t in m.TIME])
hfc_vals = np.array([m.hfc[f, t]() for f in m.FC for t in m.TIME])
pfc_fcr_up_vals = [m.pfc_fcr_up[f, t].value for f in m.FC for t in m.TIME]
pfc_fcr_dwn_vals = [m.pfc_fcr_dwn[f, t].value for f in m.FC for t in m.TIME]

pgrid_vals = [m.pgrid[t].value for t in m.TIME]   ### CORREGGERE NOME "Pgrid" in "pgrid"


#------------------------------------------------------------------------------------------------
# PROCESSING RESULTS
#------------------------------------------------------------------------------------------------

rev_tot = pyo.value(m.Obj)
rev_h2sell = sum(hdem_vals) * C_h2
rev_elecsell = sum((Ppv_vals[t] + pb_dch_vals[t] - pb_ch_vals[t] \
                - pel_vals[t] + pfc_vals[t]) * D_t * C_ele_vals[t] for t in range(time_window))
rev_FCR = sum(pel_fcr_up_vals[t] * C_FCR_capup + pel_fcr_dwn_vals[t] * C_FCR_capdown \
                + pfc_fcr_up_vals[t] * C_FCR_capup + pfc_fcr_dwn_vals[t] * C_FCR_capdown for t in range(time_window))

print(f"Total revenue = {rev_tot:.2f} €")
print(f"Revenue from selling hydrogen to the HRS = {rev_h2sell:.2f} €")
print(f"Revenue from selling excess electricity to the main grid = {rev_elecsell:.2f} €")
print(f"Revenue from participation to the FCR service = {rev_FCR:.2f} €")

In [ ]:
#------------------------------------------------------------------------------------------------
# PLOTTING (1/3)
#------------------------------------------------------------------------------------------------
time_plot = time[:time_window + 1]      # time array for plotting
day = 0                                 # specific day for plotting
week = 20                               # specific week for plotting


# Plot settings ---------------------------------------------------------------------------------
plt.rc('figure', figsize=(20, 10))
plt.rc('font', family='monospace', weight='bold', size=11)
plt.rc('axes', labelsize=11, titlesize=14)
plt.rc('legend', fontsize=11)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)
plt.rcParams['axes.grid'] = True
plt.rcParams['lines.linewidth'] = 2


# RESULTS PLOTS ---------------------------------------------------------------------------------
labels = ['H2 selling', 'Electricity selling', 'FCR participation']
colors = ["green", "blue", "orange"]
values = [rev_h2sell, rev_elecsell, rev_FCR]

plt.figure(figsize = (10, 5))
plt.bar(labels, values, color = colors)
plt.bar(['Total Revenue'], [rev_h2sell], label='Hydrogen', color = colors[0])
plt.bar(['Total Revenue'], [rev_elecsell], bottom=[rev_h2sell], label='Electricity', color = colors[1])
plt.bar(['Total Revenue'], [rev_FCR], bottom=[rev_h2sell + rev_elecsell], label='Other', color = colors[2])
plt.ylabel("Revenue [€]")
plt.grid(axis='y', linestyle='--', alpha=0.3)
# plt.tight_layout()

plt.figure(figsize=(6,5))



# plt.ylabel("Revenue (€)")
# plt.title("Revenue Composition (Stacked)")
# plt.legend()



# INPUT PLOTS -----------------------------------------------------------------------------------
# TIME WINDOW
# Electricity cost plot
figIN, axsIN = plt.subplots(3, 1, layout='constrained')
axsIN[0].stairs(C_ele_vals, time_plot, color = 'blue')
axsIN[0].set_xlabel('time [h]')
axsIN[0].set_ylabel('Electricity price [€/MWh]')
axsIN[0].set_title("Electricity price profile")
# PV power plot
axsIN[1].stairs(Ppv_vals, time_plot, color = 'blue')
axsIN[1].set_xlabel('time [h]')
axsIN[1].set_ylabel('power [MWh]')
axsIN[1].set_title("PV power generation")
# EL power plot
axsIN[2].stairs(pel_vals, time_plot, color = 'blue')
axsIN[2].set_xlabel('time [h]')
axsIN[2].set_ylabel('EL power [MW]')
axsIN[2].set_title("EL optimal power profile")
axsIN[2].set_ylim(0, 1.1 * Pel_rtd['EL'])

# CHOSEN WEEK
# Electricity cost plot
figIN2, axsIN2 = plt.subplots(3, 1, layout='constrained')
axsIN2[0].step(time_plot[week*24*7 : (week+1)*24*7], C_ele_vals[week*24*7 : (week+1)*24*7], color = 'blue')
axsIN2[0].set_xlabel('time [h]')
axsIN2[0].set_ylabel('Electricity price [€/MWh]')
axsIN2[0].set_title("Electricity price profile (week 20)")
# PV power plot
axsIN2[1].step(time_plot[week*24*7 : (week+1)*24*7], Ppv_vals[week*24*7 : (week+1)*24*7], color = 'blue')
axsIN2[1].set_xlabel('time [h]')
axsIN2[1].set_ylabel('power [MWh]')
axsIN2[1].set_title("PV power generation (week 20)")
# EL power plot
axsIN2[2].step(time_plot[week*24*7 : (week+1)*24*7], pel_vals[week*24*7 : (week+1)*24*7], color = 'blue')
axsIN2[2].set_xlabel('time [h]')
axsIN2[2].set_ylabel('EL power [MW]')
axsIN2[2].set_title("EL optimal power profile (week 20)")
axsIN2[2].set_ylim(0, 1.1 * Pel_rtd['EL'])


# EL PLOTS --------------------------------------------------------------------------------------
# TIME WINDOW
# EL power plot
figEL, axsEL = plt.subplots(3, 1, layout='constrained')
axsEL[0].stairs(pel_vals, time_plot, color = 'blue')
axsEL[0].set_xlabel('time [h]')
axsEL[0].set_ylabel('EL power [MW]')
axsEL[0].set_title("EL optimal power profile")
axsEL[0].set_ylim(0, 1.1 * Pel_rtd['EL'])
# Hydrogen production plot
axsEL[1].stairs(hel_vals, time_plot, color = 'green')
axsEL[1].set_xlabel('time [h]')
axsEL[1].set_ylabel('H2 produced [kg]')
axsEL[1].set_title("Hydrogen production")
# EL operating states plot
axsEL[2].step(time_plot[:-1], xel_on_vals, color = 'green', where='post', label = 'ON')
axsEL[2].step(time_plot[:-1], xel_off_vals, color = 'black', where='post', label = 'OFF')
axsEL[2].step(time_plot[:-1], xel_sb_vals, color = 'grey', where='post', linestyle='--', label = 'SB')
# axs[2].step(time_plot[day * 24 : (day+1) * 24], xel_su_vals[day * 24 : (day+1) * 24], color = 'red', where='post')
axsEL[2].set_xlabel('time [h]')
axsEL[2].set_ylabel('operating states [0/1]')
axsEL[2].set_title("EL operating states")
axsEL[2].legend()

# CHOSEN WEEK
# EL power plot
figEL2, axsEL2 = plt.subplots(3, 1, layout='constrained')
axsEL2[0].stairs(pel_vals[week*24*7 : (week+1)*24*7], time_plot[week*24*7 : (week+1)*24*7 + 1], color = 'blue')
axsEL2[0].set_xlabel('time [h]')
axsEL2[0].set_ylabel('EL power [MW]')
axsEL2[0].set_title("EL optimal power profile (week 20)")
axsEL2[0].set_xlim(week*24*7, (week+1)*24*7 + 1)
axsEL2[0].set_ylim(0, 1.1 * Pel_rtd['EL'])
# Hydrogen production plot
axsEL2[1].stairs(hel_vals[week*24*7 : (week+1)*24*7], time_plot[week*24*7 : (week+1)*24*7 + 1], color = 'green')
axsEL2[1].set_xlabel('time [h]')
axsEL2[1].set_ylabel('H2 produced [kg]')
axsEL2[1].set_title("Hydrogen production (week 20)")
axsEL2[1].set_xlim(week*24*7, (week+1)*24*7 + 1)
# EL operating states plot
axsEL2[2].step(time_plot[week*24*7 : (week+1)*24*7], xel_on_vals[week*24*7 : (week+1)*24*7], color = 'green', where='post', label = 'ON')
axsEL2[2].step(time_plot[week*24*7 : (week+1)*24*7], xel_off_vals[week*24*7 : (week+1)*24*7], color = 'black', where='post', label = 'OFF')
axsEL2[2].step(time_plot[week*24*7 : (week+1)*24*7], xel_sb_vals[week*24*7 : (week+1)*24*7], color = 'grey', where='post', linestyle='--', label = 'SB')
# axs[2].step(time_plot[day * 24 : (day+1) * 24], xel_su_vals[day * 24 : (day+1) * 24], color = 'red', where='post')
axsEL2[2].set_xlabel('time [h]')
axsEL2[2].set_ylabel('operating states [0/1]')
axsEL2[2].set_title("EL operating states (week 20)")
axsEL2[2].set_xlim(week*24*7, (week+1)*24*7 + 1)
axsEL2[2].legend()

In [ ]:
#------------------------------------------------------------------------------------------------
# PLOTTING (2/3)
#------------------------------------------------------------------------------------------------

week2 = 34

# BESS PLOTS ------------------------------------------------------------------------------------
# TIME WINDOW
# BESS power plot
figBESS, axsBESS = plt.subplots(2, 1, layout='constrained')
axsBESS[0].stairs(pb_vals, time_plot, color = 'blue')
axsBESS[0].set_xlabel('time [h]')
axsBESS[0].set_ylabel('BESS power [MW]')
axsBESS[0].set_title("BESS optimal power profile")
# BESS SoC plot
axsBESS[1].plot(time_plot, soc_vals, color = 'black')
axsBESS[1].set_xlabel('time [h]')
axsBESS[1].set_ylabel('SoC [p.u.]')
axsBESS[1].set_title("BESS State of Charge")
axsBESS[1].set_ylim(0, 1.1)
# # BESS operating states plot
# axs2[2].step(time_plot[week*24*7 : (week+1)*24*7], pb_dch_vals[week*24*7 : (week+1)*24*7] - pb_ch_vals[week*24*7 : (week+1)*24*7], color = 'blue')
# axs2[2].step(time_plot[week*24*7 : (week+1)*24*7], 0.5*xb_vals[week*24*7 : (week+1)*24*7], color = 'black')

# CHOSEN WEEK
# BESS power plot
figBESS2, axsBESS2 = plt.subplots(2, 1, layout='constrained')
axsBESS2[0].step(time_plot[week*24*7 : (week+1)*24*7], pb_vals[week*24*7 : (week+1)*24*7], color = 'blue')
axsBESS2[0].set_xlabel('time [h]')
axsBESS2[0].set_ylabel('BESS power [MW]')
axsBESS2[0].set_title("BESS optimal power profile (week 20)")
axsBESS2[0].set_xlim(week*24*7, (week+1)*24*7 + 1)
# BESS SoC plot
axsBESS2[1].plot(time_plot[week*24*7 : (week+1)*24*7], soc_vals[week*24*7 + 1 : (week+1)*24*7 + 1], color = 'black')  # SoC values are shifted forwards by 1 to match power plots
axsBESS2[1].set_xlabel('time [h]')
axsBESS2[1].set_ylabel('SoC [p.u.]')
axsBESS2[1].set_title("BESS State of Charge (week 20)")
axsBESS2[1].set_ylim(0, 1.1)
axsBESS2[1].set_xlim(week*24*7, (week+1)*24*7 + 1)


# HYDROGEN STORAGE PLOTS ------------------------------------------------------------------------
# TIME WINDOW
# HSS hydrogen level plot
figHSS, axsHSS = plt.subplots(2, 1, layout='constrained')
axsHSS[0].plot(time_plot, hhss_vals, color = 'black')
axsHSS[0].set_xlabel('time [h]')
axsHSS[0].set_ylabel('hydrogen level [kg]')
axsHSS[0].set_title("Hydrogen level in the storage tank")
axsHSS[0].set_ylim(0, 1.1 * H_hss_max['HSS'])
# Input and outputs of the HSS (without efficiencies)
axsHSS[1].step(time_plot[:-1], hel_vals, color = 'green', label = 'H2 from EL')
axsHSS[1].step(time_plot[:-1], hfc_vals, color = 'red', label = 'H2 to FC')
axsHSS[1].step(time_plot[:-1], hdem_vals, color = 'black', label = 'H2 to HRS')
axsHSS[1].set_xlabel('time [h]')
axsHSS[1].set_ylabel('hydrogen flows [kg]')
axsHSS[1].set_title("Storage tank input and output hydrogen flows")
axsHSS[1].set_ylim(0, 40)
axsHSS[1].legend()

# CHOSEN WEEK
# HSS hydrogen level plot
figHSS2, axsHSS2 = plt.subplots(2, 1, layout='constrained')
axsHSS2[0].plot(time_plot[week*24*7 : (week+1)*24*7], hhss_vals[week*24*7 + 1 : (week+1)*24*7 + 1], color = 'black')
axsHSS2[0].set_xlabel('time [h]')
axsHSS2[0].set_ylabel('hydrogen level [kg]')
axsHSS2[0].set_title("Hydrogen level in the storage tank (week 20)")
axsHSS2[0].set_ylim(0.95 * min(hhss_vals[week*24*7 + 1 : (week+1)*24*7 + 1]), 1.05 * max(hhss_vals[week*24*7 + 1 : (week+1)*24*7 + 1]))
axsHSS2[0].set_xlim(week*24*7, (week+1)*24*7 + 1)
# Input and outputs of the HSS (without efficiencies)
axsHSS2[1].step(time_plot[week*24*7 : (week+1)*24*7], hel_vals[week*24*7 : (week+1)*24*7], color = 'green', label = 'H2 from EL')
axsHSS2[1].step(time_plot[week*24*7 : (week+1)*24*7], hfc_vals[week*24*7 : (week+1)*24*7], color = 'red', label = 'H2 to FC')
axsHSS2[1].step(time_plot[week*24*7 : (week+1)*24*7], hdem_vals[week*24*7 : (week+1)*24*7], color = 'black', label = 'H2 to HRS')
axsHSS2[1].set_xlabel('time [h]')
axsHSS2[1].set_ylabel('hydrogen flows [kg]')
axsHSS2[1].set_title("Storage tank input and output hydrogen flows (week 20)")
axsHSS2[1].set_ylim(0, 40)
axsHSS2[1].set_xlim(week*24*7, (week+1)*24*7 + 1)
axsHSS2[1].legend()

# CHOSEN WEEK 2
# HSS hydrogen level plot
figHSS3, axsHSS3 = plt.subplots(2, 1, layout='constrained')
axsHSS3[0].plot(time_plot[week2*24*7 : (week2+1)*24*7], hhss_vals[week2*24*7 + 1 : (week2+1)*24*7 + 1], color = 'black')
axsHSS3[0].set_xlabel('time [h]')
axsHSS3[0].set_ylabel('hydrogen level [kg]')
axsHSS3[0].set_title("Hydrogen level in the storage tank (week 34)")
axsHSS3[0].set_xlim(week2*24*7, (week2+1)*24*7 + 1)
axsHSS3[0].set_ylim(0.95 * min(hhss_vals[week2*24*7 + 1 : (week2+1)*24*7 + 1]), 1.05 * max(hhss_vals[week2*24*7 + 1 : (week2+1)*24*7 + 1]))
# Input and outputs of the HSS (without efficiencies)
axsHSS3[1].step(time_plot[week2*24*7 : (week2+1)*24*7], hel_vals[week2*24*7 : (week2+1)*24*7], color = 'green', label = 'H2 from EL')
axsHSS3[1].step(time_plot[week2*24*7 : (week2+1)*24*7], hfc_vals[week2*24*7 : (week2+1)*24*7], color = 'red', label = 'H2 to FC')
axsHSS3[1].step(time_plot[week2*24*7 : (week2+1)*24*7], hdem_vals[week2*24*7 : (week2+1)*24*7], color = 'black', label = 'H2 to HRS')
axsHSS3[1].set_xlabel('time [h]')
axsHSS3[1].set_ylabel('hydrogen flows [kg]')
axsHSS3[1].set_title("Storage tank input and output hydrogen flows (week 34)")
axsHSS3[1].set_ylim(0, 40)
axsHSS3[1].set_xlim(week2*24*7, (week2+1)*24*7 + 1)
axsHSS3[1].legend()


# FC power plot
figFC, axsFC = plt.subplots(2, 1, layout='constrained')
axsFC[0].stairs(pfc_vals, time_plot, color = 'red')
axsFC[0].set_xlabel('time [h]')
axsFC[0].set_ylabel('FC power [MW]')
axsFC[0].set_title("FC optimal power profile")
axsFC[0].set_ylim(0, 1.1 * Pfc_rtd['FC'])

In [ ]:
#------------------------------------------------------------------------------------------------
# PLOTTING (3/3)
#------------------------------------------------------------------------------------------------

# FCR EL PLOTS ----------------------------------------------------------------------------------
# TIME WINDOW
# FCR EL capacity allocation plot, downward regulation
figFCRel, axsFCRel = plt.subplots(2, 1, layout = 'constrained')
axsFCRel[0].step(time_plot[:-1], pel_vals + pel_fcr_dwn_vals, color = 'blue', label = 'EL power + downward FCR')
axsFCRel[0].step(time_plot[:-1], pel_vals, color = 'black', label = 'EL power')
axsFCRel[0].set_xlabel('time [h]')
axsFCRel[0].set_ylabel('allocated FCR capacity [MW]')
axsFCRel[0].set_title("Downward FCR regulation of the EL")
axsFCRel[0].set_ylim(-0.1, 1.1 * Pel_rtd['EL'])
axsFCRel[0].legend()
# FCR EL capacity allocation plot, upward regulation
axsFCRel[1].step(time_plot[:-1], pel_vals - pel_fcr_up_vals, color = 'blue', label = 'EL power - upward FCR')
axsFCRel[1].step(time_plot[:-1], pel_vals, color = 'black', label = 'EL power')
axsFCRel[1].set_xlabel('time [h]')
axsFCRel[1].set_ylabel('allocated FCR capacity [MW]')
axsFCRel[1].set_title("Upward FCR regulation of the EL")
axsFCRel[1].set_ylim(-0.1, 1.1 * Pel_rtd['EL'])
axsFCRel[1].legend()

# CHOSEN WEEK
# FCR EL capacity allocation plot, downward regulation
figFCRel, axsFCRel = plt.subplots(2, 1, layout = 'constrained')
axsFCRel[0].step(time_plot[week*24*7 : (week+1)*24*7], pel_vals[week*24*7 : (week+1)*24*7] + pel_fcr_dwn_vals[week*24*7 : (week+1)*24*7], color = 'blue', label = 'EL power + downward FCR')
axsFCRel[0].step(time_plot[week*24*7 : (week+1)*24*7], pel_vals[week*24*7 : (week+1)*24*7], color = 'black', label = 'EL power')
axsFCRel[0].set_xlabel('time [h]')
axsFCRel[0].set_ylabel('allocated FCR capacity [MW]')
axsFCRel[0].set_title("Downward FCR regulation of the EL (week 20)")
axsFCRel[0].set_ylim(-0.1, 1.1 * Pel_rtd['EL'])
axsFCRel[0].legend()
# FCR EL capacity allocation plot, upward regulation
axsFCRel[1].step(time_plot[week*24*7 : (week+1)*24*7], pel_vals[week*24*7 : (week+1)*24*7] - pel_fcr_up_vals[week*24*7 : (week+1)*24*7], color = 'blue', label = 'EL power - upward FCR')
axsFCRel[1].step(time_plot[week*24*7 : (week+1)*24*7], pel_vals[week*24*7 : (week+1)*24*7], color = 'black', label = 'EL power')
axsFCRel[1].set_xlabel('time [h]')
axsFCRel[1].set_ylabel('allocated FCR capacity [MW]')
axsFCRel[1].set_title("Upward FCR regulation of the EL (week 20)")
axsFCRel[1].set_ylim(-0.1, 1.1 * Pel_rtd['EL'])
axsFCRel[1].legend()

In [ ]:
20*7*24 + 168